# Lab 3: Building and Training Neural Networks

## 🎯 Learning Objectives

By the end of this lab, you will:
- Understand how neural network components (Neuron, Layer, MLP) work
- Implement a complete Multi-Layer Perceptron (MLP) from scratch
- Implement a loss function with L2 regularization
- Train a neural network using gradient descent
- Visualize decision boundaries to validate your model

**What You'll Build:** A 2-layer neural network that classifies 2D data (same as [Karpathy's micrograd demo](https://github.com/karpathy/micrograd/blob/master/demo.ipynb))

**Note:** All packages are pre-installed in Google Colab!

**Setup:** Run this cell first to download required files

In [ ]:
# Download required files from GitHub (for Colab users)
import sys
import os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Download engine.py (complete Value class with autograd)
    !wget -q https://raw.githubusercontent.com/hhe0u0/micrograd-lab/claude/micrograd-lab-exercises-V14m2/engine.py
    # Download utils.py (visualization functions)
    !wget -q https://raw.githubusercontent.com/hhe0u0/micrograd-lab/claude/micrograd-lab-exercises-V14m2/utils.py
    print("✓ Helper files downloaded (engine.py, utils.py)!")
else:
    if os.path.exists('engine.py') and os.path.exists('utils.py'):
        print("✓ Helper files found locally!")
    else:
        print("⚠️  engine.py or utils.py not found. Make sure you're in the micrograd-lab directory.")

## Part 1: The Engine - Complete Value Class with Autograd

### You Get a Complete Autograd Engine!

Instead of implementing the autograd engine from scratch, we provide the complete `Value` class from micrograd. This lets you focus on building and training neural networks.

**Reference:** [micrograd/engine.py](https://github.com/karpathy/micrograd/blob/master/micrograd/engine.py)

### Key Features:

1. **Forward pass:** Tracks all operations (+, *, **, relu, tanh)
2. **Backward pass:** Automatically computes gradients via `backward()` method
3. **Gradient accumulation:** Handles cases where a value is used multiple times

Let's import and explore it:

In [ ]:
# Import the complete Value class
from engine import Value

# Demo: Simple expression
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

e = a * b
e.label = 'e'
d = e + c
d.label = 'd'
f = d.relu()  # ReLU activation: max(0, d)
f.label = 'f'

print(f"Forward pass:")
print(f"a = {a.data}")
print(f"b = {b.data}")
print(f"e = a * b = {e.data}")
print(f"d = e + c = {d.data}")
print(f"f = ReLU(d) = {f.data}")

### Automatic Differentiation in Action

Call `backward()` to automatically compute all gradients:

In [ ]:
# Compute gradients automatically!
f.backward()

print("\nAfter backward():")
print(f"a.grad = {a.grad}  (df/da)")
print(f"b.grad = {b.grad}  (df/db)")
print(f"c.grad = {c.grad}  (df/dc)")
print(f"\n✓ Gradients computed automatically using chain rule!")

### Visualize the Computation Graph

Let's see the full graph with gradients:

In [ ]:
from utils import visualize_graph

visualize_graph(f, title="Computation Graph with Gradients")

**What you should see:**
- All nodes with their data AND grad values
- ReLU operation node
- Gradients flowing backwards from f to inputs
- Notice how gradients are computed automatically!

## Part 2: Building Neural Network Components

### Neural Network Architecture

A neural network is built from these components:

```
Neuron: Single computation unit
  ↓
Layer: Collection of Neurons
  ↓
MLP (Multi-Layer Perceptron): Stack of Layers
```

### What is a Neuron?

A neuron computes: **output = activation(w0*x0 + w1*x1 + ... + wn*xn + bias)**

Where:
- **Inputs:** x0, x1, ..., xn (from data)
- **Weights:** w0, w1, ..., wn (learned parameters)
- **Bias:** b (learned parameter)
- **Activation:** ReLU or tanh (adds non-linearity)

Example:
```python
# 2-input neuron
inputs = [2.0, 3.0]
weights = [Value(0.5), Value(-0.3)]
bias = Value(1.0)

output = (weights[0]*inputs[0] + weights[1]*inputs[1] + bias).relu()
# output = (0.5*2.0 + (-0.3)*3.0 + 1.0).relu()
# output = (1.0 - 0.9 + 1.0).relu()
# output = ReLU(1.1) = 1.1
```

## Exercise 1: Implement Neural Network Components

### Your Task

Implement the `__call__` method for:
1. **Neuron**: Computes weighted sum + bias + activation
2. **Layer**: Applies multiple neurons to same input
3. **MLP**: Stacks multiple layers

**Reference:** [micrograd/nn.py](https://github.com/karpathy/micrograd/blob/master/micrograd/nn.py)

### Starter Code

In [ ]:
import random
from engine import Value

class Module:
    """Base class for all neural network components."""
    
    def zero_grad(self):
        """Reset all gradients to zero."""
        for p in self.parameters():
            p.grad = 0.0
    
    def parameters(self):
        return []


class Neuron(Module):
    """A single neuron with nin inputs."""
    
    def __init__(self, nin, nonlin=True):
        """
        Args:
            nin: Number of inputs
            nonlin: If True, apply ReLU activation. If False, linear (no activation)
        """
        # Initialize weights randomly between -1 and 1
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)  # Bias initialized to 0
        self.nonlin = nonlin
    
    def __call__(self, x):
        """
        Forward pass through neuron.
        
        Args:
            x: List of input values (can be regular floats or Value objects)
        
        Returns:
            Value object representing neuron output
        
        1. Compute weighted sum: w0*x0 + w1*x1 + ... + wn*xn + b
           Hint: sum([wi*xi for wi, xi in zip(self.w, x)], self.b)
        2. Apply activation:
           - If self.nonlin is True: return act.relu()
           - If self.nonlin is False: return act (linear)
        """
        raise NotImplementedError("Implement Neuron.__call__")
    
    def parameters(self):
        """Return all parameters (weights + bias)."""
        return self.w + [self.b]
    
    def __repr__(self):
        return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"


class Layer(Module):
    """A layer of neurons."""
    
    def __init__(self, nin, nout, **kwargs):
        """
        Args:
            nin: Number of inputs
            nout: Number of neurons in this layer
            **kwargs: Passed to Neuron (e.g., nonlin=False for output layer)
        """
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]
    
    def __call__(self, x):
        """
        Forward pass through layer.
        
        Args:
            x: List of input values
        
        Returns:
            List of outputs (one per neuron), or single Value if only 1 neuron
        
        1. Call each neuron in self.neurons with input x
           Hint: out = [neuron(x) for neuron in self.neurons]
        2. If only 1 neuron, return out[0] (single Value)
        3. Otherwise return out (list of Values)
        """
        raise NotImplementedError("Implement Layer.__call__")
    
    def parameters(self):
        """Return all parameters from all neurons."""
        return [p for n in self.neurons for p in n.parameters()]
    
    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"


class MLP(Module):
    """Multi-Layer Perceptron (stack of layers)."""
    
    def __init__(self, nin, nouts):
        """
        Args:
            nin: Number of inputs
            nouts: List of layer sizes
                   Example: MLP(2, [16, 16, 1]) creates:
                   - Layer 1: 2 inputs → 16 neurons (ReLU)
                   - Layer 2: 16 inputs → 16 neurons (ReLU)
                   - Layer 3: 16 inputs → 1 neuron (Linear)
        """
        sz = [nin] + nouts
        # Create layers, last layer has no activation (nonlin=False)
        self.layers = [
            Layer(sz[i], sz[i+1], nonlin=(i != len(nouts)-1))
            for i in range(len(nouts))
        ]
    
    def __call__(self, x):
        """
        Forward pass through entire network.
        
        Args:
            x: Input (list of values)
        
        Returns:
            Output of final layer
        
        1. Pass input x through first layer
        2. Pass result through second layer
        3. Continue through all layers
        4. Return final output
        
        Hint: Use a loop to apply each layer sequentially:
        for layer in self.layers:
            x = layer(x)
        return x
        """
        raise NotImplementedError("Implement MLP.__call__")
    
    def parameters(self):
        """Return all parameters from all layers."""
        return [p for layer in self.layers for p in layer.parameters()]
    
    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"

### Test Your Neural Network Components

In [ ]:
# Test 1: Create a single neuron
n = Neuron(2)  # 2 inputs
x = [0.5, -1.0]
output = n(x)

print("Test 1: Single Neuron")
print(f"Neuron: {n}")
print(f"Input: {x}")
print(f"Output: {output}")
print(f"Number of parameters: {len(n.parameters())}  (should be 3: 2 weights + 1 bias)")
assert len(n.parameters()) == 3
print("✓ Neuron: PASS\n")

In [ ]:
# Test 2: Create a layer with 3 neurons
layer = Layer(2, 3)  # 2 inputs, 3 neurons
x = [0.5, -1.0]
outputs = layer(x)

print("Test 2: Layer")
print(f"Layer: {layer}")
print(f"Input: {x}")
print(f"Outputs: {outputs}  (list of 3 Values)")
print(f"Number of parameters: {len(layer.parameters())}  (should be 9: 3 neurons × 3 params)")
assert len(outputs) == 3
assert len(layer.parameters()) == 9
print("✓ Layer: PASS\n")

In [ ]:
# Test 3: Create a complete MLP
model = MLP(2, [4, 4, 1])  # 2 inputs → 4 neurons → 4 neurons → 1 output
x = [0.5, -1.0]
output = model(x)

print("Test 3: MLP")
print(f"Model: {model}")
print(f"Input: {x}")
print(f"Output: {output}  (single Value)")
print(f"Number of parameters: {len(model.parameters())}")
# (2+1)*4 + (4+1)*4 + (4+1)*1 = 12 + 20 + 5 = 37
assert len(model.parameters()) == 37
print("✓ MLP: PASS\n")

print("🎉 All tests passed! Your neural network is ready.")

## Part 3: Preparing the Dataset

### The Problem: Binary Classification

We'll solve the same problem as Karpathy's demo: classify 2D points into two classes (moon-shaped data).

**Dataset:** make_moons from scikit-learn
- 100 samples
- 2 features (x, y coordinates)
- 2 classes: -1 and +1

Let's load and visualize the data:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# Set seeds for reproducibility
np.random.seed(1337)
random.seed(1337)

# Create dataset
X, y = make_moons(n_samples=100, noise=0.1)
y = y*2 - 1  # Convert labels from {0, 1} to {-1, +1}

# Visualize
plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], c=y, s=20, cmap='jet')
plt.title("Training Data (Moon Dataset)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"Classes: {np.unique(y)}  (red=-1, blue=+1)")

**What you should see:**
- Two moon-shaped clusters (red and blue)
- This is NOT linearly separable (can't separate with a line)
- Need a neural network to classify it!

## Exercise 2: Implement Loss Function

### Loss Function Components

Our loss function has two parts:

1. **Data Loss (SVM max-margin loss):**
   - For each sample: `loss = max(0, 1 - y_true * y_pred)`
   - This is called "hinge loss"
   - Using ReLU: `loss = (1 - y_true * y_pred).relu()`

2. **Regularization Loss (L2):**
   - Penalizes large weights to prevent overfitting
   - `reg_loss = alpha * sum(w^2 for all weights)`
   - alpha is regularization strength (typically 1e-4)

**Total Loss** = Data Loss + Regularization Loss

### Your Task

Implement the `loss()` function following [demo.ipynb](https://github.com/karpathy/micrograd/blob/master/demo.ipynb):

In [ ]:
def loss(model, X, y, alpha=1e-4):
    """
    Compute total loss (data loss + regularization).
    
    Args:
        model: MLP model
        X: Input data (numpy array of shape [n_samples, n_features])
        y: True labels (numpy array of shape [n_samples])
        alpha: Regularization strength
    
    Returns:
        total_loss: Value object
        accuracy: Float (percentage correct)
    
    1. Convert inputs to Value objects:
       inputs = [list(map(Value, xrow)) for xrow in X]
    
    2. Forward pass - get predictions:
       scores = [model(x) for x in inputs]
    
    3. Compute SVM max-margin loss:
       losses = [(1 + -yi*scorei).relu() for yi, scorei in zip(y, scores)]
       data_loss = sum(losses) * (1.0 / len(losses))
    
    4. Compute L2 regularization:
       reg_loss = alpha * sum((p*p for p in model.parameters()))
    
    5. Total loss:
       total_loss = data_loss + reg_loss
    
    6. Compute accuracy:
       accuracy = [(yi > 0) == (scorei.data > 0) for yi, scorei in zip(y, scores)]
       accuracy = sum(accuracy) / len(accuracy)
    
    7. Return total_loss, accuracy
    """
    raise NotImplementedError("Implement loss function")

### Test Your Loss Function

In [ ]:
# Create a model
model = MLP(2, [16, 16, 1])  # 2 inputs → 16 → 16 → 1 output
print(f"Model: {model}")
print(f"Number of parameters: {len(model.parameters())}")

# Compute initial loss
total_loss, acc = loss(model, X, y)
print(f"\nInitial loss: {total_loss.data:.4f}")
print(f"Initial accuracy: {acc*100:.1f}%")
print("\n✓ Loss function works!")

## Exercise 3: Implement Training Loop

### The Training Algorithm

Training a neural network follows this pattern:

```
for each training step:
    1. Forward pass: compute loss
    2. Backward pass: compute gradients
    3. Update parameters: w = w - learning_rate * gradient
    4. Zero gradients for next iteration
```

### Key Concepts

**Learning Rate:** How big a step to take when updating weights
- Too large: Training unstable, might diverge
- Too small: Training too slow
- Typical: 0.01 to 0.1

**Gradient Descent:** Move weights in opposite direction of gradient (downhill)

### Your Task

Implement the training loop following [demo.ipynb](https://github.com/karpathy/micrograd/blob/master/demo.ipynb):

In [ ]:
from tqdm import tqdm

# Training hyperparameters
learning_rate = 0.1  # Increased from 0.05 for faster convergence
num_steps = 50  # Reduced from 100 for faster training

print("\nUncomment the code above to train the model!")

### What You Should See

During training:
- Loss should decrease (not always monotonically)
- Accuracy should increase
- Final accuracy should be > 90%

Example output:
```
step 0 loss 1.2345, accuracy 52.0%
step 10 loss 0.8432, accuracy 75.0%
step 20 loss 0.4521, accuracy 85.0%
...
step 90 loss 0.1234, accuracy 97.0%
```

## Part 4: Visualizing Decision Boundaries

### Why Visualize?

Decision boundaries show:
- How well the model separates classes
- Whether the model overfits or underfits
- Which regions the model is confident about

### Decision Boundary Visualization Function

We provide a helper function that:
1. Creates a grid of points
2. Predicts class for each point
3. Colors the background based on predictions
4. Overlays the actual data points

In [ ]:
def plot_decision_boundary(model, X, y):
    """
    Visualize decision boundary of the model.
    
    Args:
        model: Trained MLP model
        X: Input data (numpy array)
        y: True labels (numpy array)
    """
    # Create a mesh grid
    h = 0.25
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict for each point in mesh
    Xmesh = np.c_[xx.ravel(), yy.ravel()]
    inputs = [list(map(Value, xrow)) for xrow in Xmesh]
    scores = [model(x) for x in inputs]
    scores = np.array([s.data for s in scores])
    scores = scores.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(5, 5))
    plt.contourf(xx, yy, scores, levels=20, cmap='RdYlBu', alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap='jet', edgecolors='black', linewidth=1)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title('Decision Boundary')
    plt.colorbar(label='Model Output')
    plt.show()

# Try it (after training):
# plot_decision_boundary(model, X, y)

### Expected Result

After training, you should see:
- **Background colors:** Show model's predictions across the space
  - Blue region: Model predicts class -1
  - Red region: Model predicts class +1
- **Data points:** Actual samples (red and blue)
- **Smooth boundary:** Separating the two moons

If your model is working correctly, the decision boundary should cleanly separate the two moon shapes!

## Putting It All Together

### Complete Training Pipeline

Run this cell after implementing all exercises:

In [ ]:
# Uncomment to run complete pipeline:

# # 1. Create model
# model = MLP(2, [16, 16, 1])
# print(f"Model created with {len(model.parameters())} parameters\n")

# # 2. Train
# for step in range(100):
#     # Forward
#     total_loss, acc = loss(model, X, y)
#     
#     # Backward
#     model.zero_grad()
#     total_loss.backward()
#     
#     # Update
#     for p in model.parameters():
#         p.data += -0.05 * p.grad
#     
#     if step % 10 == 0:
#         print(f"step {step} loss {total_loss.data:.4f}, accuracy {acc*100:.1f}%")

# # 3. Visualize
# plot_decision_boundary(model, X, y)

# print("\n🎉 Training complete! Check the decision boundary above.")

## Summary

### What You've Built

✅ **Complete neural network** (Neuron, Layer, MLP)

✅ **Loss function** with SVM max-margin loss

✅ **L2 regularization** to prevent overfitting

✅ **Training loop** with gradient descent

✅ **Decision boundary visualization** to validate results

### Key Concepts

- **Forward pass:** Data flows through layers to produce predictions
- **Backward pass:** Gradients flow backwards to update weights
- **Activation functions:** Add non-linearity (ReLU)
- **Loss function:** Measures how wrong predictions are
- **Regularization:** Prevents overfitting by penalizing large weights
- **Gradient descent:** Iteratively improves weights

### You've Built a Complete Deep Learning System!

This is the **exact same architecture** used in production frameworks like PyTorch:
- Computation graphs track operations ✓
- Automatic differentiation computes gradients ✓
- Modular components (Neuron, Layer, MLP) ✓
- Training loop optimizes weights ✓

### Next Steps

In **Lab 4**, you'll:
- Use **PyTorch** (production deep learning framework)
- Train on **MNIST** (real handwritten digits dataset)
- Implement complete training and validation pipeline
- See how PyTorch makes this easier (but works the same way!)

**Congratulations!** 🎉 You've built a neural network from scratch and trained it successfully. You now understand the fundamentals of how deep learning works!